# Perplexity: hands-on (optional)
**LLMs From Basics · Lesson M4.1**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (plus a ✓) means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*

Only `numpy` and `matplotlib` are needed (both pre-installed on Colab).

In [ ]:
import math, re
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("ready")

## 1 · Perplexity from token probabilities
A model reads held-out text and gives the actual next token probability $p_t$ at each of $N$ positions. Perplexity is

$$\text{PPL} = \exp\Big(-\frac{1}{N}\sum_{t=1}^{N} \ln p_t\Big)$$

Implement it with natural logs.

In [ ]:
def perplexity(probs):
    probs = np.asarray(probs, dtype=float)
    # TODO: mean of -ln(p), then exponentiate
    raise NotImplementedError("Exercise 1")

**Check 1.** Uniform over 4 options must give exactly 4; the lesson's worked example gives about 3.16.

In [ ]:
assert abs(perplexity([0.25] * 10) - 4) < 1e-9
assert abs(perplexity([0.5, 0.25, 0.8, 0.1]) - 3.1623) < 1e-3
assert abs(perplexity([1.0, 1.0]) - 1) < 1e-12, "perfect prediction → PPL 1"
# geometric mean of 1/p is the same thing
p = np.random.uniform(0.05, 1, 50)
assert abs(perplexity(p) - np.prod(1 / p) ** (1 / len(p))) < 1e-9
print("✓ exercise 1")

## 2 · Nats, bits and the loss
Training logs report the cross-entropy loss, usually in **nats** (natural log). Some papers use **bits** (log base 2). Implement:
- `loss_to_ppl(loss_nats)`: perplexity from a loss in nats,
- `bits_per_token(probs)`: the mean surprise $-\log_2 p$ in bits.

Then $2^{\text{bits}}$ should equal the perplexity.

In [ ]:
def loss_to_ppl(loss_nats):
    # TODO
    raise NotImplementedError("Exercise 2a")

def bits_per_token(probs):
    # TODO: mean of -log2(p)
    raise NotImplementedError("Exercise 2b")

**Check 2**

In [ ]:
assert abs(loss_to_ppl(2.0) - 7.389) < 1e-3
assert abs(loss_to_ppl(math.log(2)) - 2) < 1e-12
probs = [0.1, 0.5, 0.2]
assert abs(2 ** bits_per_token(probs) - perplexity(probs)) < 1e-9, "same number, different log base"
print("✓ exercise 2: PPL =", round(perplexity(probs), 3))

## 3 · A tiny bigram language model
Real LLMs condition on thousands of tokens. A **bigram** model conditions on just one: $p(x_t \mid x_{t-1})$. We estimate it from counts with **add-α smoothing**, so unseen pairs still get a little probability:

$$p(j \mid i) = \frac{\text{count}(i \to j) + \alpha}{\sum_{j'} \text{count}(i \to j') + \alpha V}$$

where $V$ is the vocabulary size. Words never seen in training become `<unk>`.

In [ ]:
train_text = '''
the cat sat on the mat . the dog sat on the rug . the cat saw the dog .
the dog saw the cat . a cat ran to the mat . a dog ran to the rug .
the cat is on the mat . the dog is on the rug . the bird sat on the cat .
a bird saw a dog . the dog ran to the bird . the cat ran to the dog .
the bird is on the mat . a cat sat on a rug . the dog sat on a mat .
'''
heldout_text = '''
the bird sat on the rug . a dog saw the bird . the cat is on a rug .
the fox ran to the cat . a bird ran to the mat .
'''

def tokenize(text):
    return re.findall(r"[a-z]+|[.]", text.lower())

train_tok, held_tok = tokenize(train_text), tokenize(heldout_text)
vocab = sorted(set(train_tok)) + ["<unk>"]
idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
def to_ids(tokens):
    return np.array([idx.get(w, idx["<unk>"]) for w in tokens])
train_ids, held_ids = to_ids(train_tok), to_ids(held_tok)

counts = np.zeros((V, V))
for a, b in zip(train_ids[:-1], train_ids[1:]):
    counts[a, b] += 1
print(V, "words in vocab;", len(train_ids), "training tokens;", len(held_ids), "held-out tokens")

In [ ]:
def bigram_probs(counts, alpha):
    # TODO: return a V×V matrix P with P[i, j] = p(j | i), using add-alpha smoothing
    raise NotImplementedError("Exercise 3")

**Check 3**

In [ ]:
P = bigram_probs(counts, alpha=0.1)
assert P.shape == (V, V)
assert np.allclose(P.sum(axis=1), 1), "each row is a probability distribution"
assert (P > 0).all(), "smoothing gives every pair some probability"
assert P[idx["the"], idx["cat"]] > P[idx["the"], idx["sat"]], "'the cat' is common, 'the sat' never happens"
print("✓ exercise 3")

## 4 · Train vs held-out perplexity
Score a token sequence with the bigram model: the model predicts every token after the first, so there are $N - 1$ predictions. Return the perplexity over them.

In [ ]:
def sequence_ppl(P, ids):
    # TODO: probabilities P[prev, next] for each consecutive pair, then perplexity
    raise NotImplementedError("Exercise 4")

**Check 4.** A model always fits its own training text better than new text. And a *uniform* model, which gives every word $1/V$, has perplexity exactly $V$.

In [ ]:
ppl_train, ppl_held = sequence_ppl(P, train_ids), sequence_ppl(P, held_ids)
print(f"train PPL {ppl_train:.2f}   held-out PPL {ppl_held:.2f}   vocab size {V}")
assert ppl_train < ppl_held < V
uniform = np.full((V, V), 1 / V)
assert abs(sequence_ppl(uniform, held_ids) - V) < 1e-9
print("✓ exercise 4")

## 5 · The tokenizer trap
Perplexity is *per token*. Two models give the same 100-byte text a total surprise of 50 nats, but model A's tokenizer uses 25 tokens and model B's uses 40. Implement both measures and compare.

In [ ]:
def per_token_ppl(total_nll_nats, n_tokens):
    # TODO
    raise NotImplementedError("Exercise 5a")

def bits_per_byte(total_nll_nats, n_bytes):
    # TODO: convert nats to bits, divide by bytes
    raise NotImplementedError("Exercise 5b")

**Check 5**

In [ ]:
a, b = per_token_ppl(50, 25), per_token_ppl(50, 40)
print(f"per-token PPL: A {a:.2f}, B {b:.2f}  |  bits per byte: {bits_per_byte(50, 100):.3f} for both")
assert abs(a - 7.389) < 1e-3 and abs(b - 3.490) < 1e-3
assert abs(bits_per_byte(50, 100) - 0.7213) < 1e-3
print("✓ exercise 5: same text likelihood, very different per-token perplexity")

## Experiment
1. **Smoothing.** Plot train and held-out perplexity as α varies. Tiny α trusts the counts (great on train, bad on new text); huge α flattens everything toward uniform.
2. **Characters instead of words.** Build a *character* bigram model on the same text and print both per-token perplexities. They are on different scales: one model predicts a whole word per step, the other a single character, so comparing the two numbers says nothing about which model is better. That's the tokenizer trap again.

In [ ]:
alphas = [0.001, 0.01, 0.03, 0.1, 0.3, 1, 3]
tr = [sequence_ppl(bigram_probs(counts, a), train_ids) for a in alphas]
he = [sequence_ppl(bigram_probs(counts, a), held_ids) for a in alphas]
plt.figure(figsize=(7, 4))
plt.plot(alphas, tr, "o-", label="train"); plt.plot(alphas, he, "o-", label="held-out")
plt.xscale("log"); plt.xlabel("smoothing α"); plt.ylabel("perplexity"); plt.legend()
plt.title("Bigram model: train vs held-out perplexity"); plt.show()

chars = sorted(set(train_text) | set(heldout_text))
cidx = {ch: i for i, ch in enumerate(chars)}
c_train = np.array([cidx[ch] for ch in train_text]); c_held = np.array([cidx[ch] for ch in heldout_text])
cc = np.zeros((len(chars), len(chars)))
for a_, b_ in zip(c_train[:-1], c_train[1:]):
    cc[a_, b_] += 1
print("word-level held-out PPL per token:", round(sequence_ppl(bigram_probs(counts, 0.1), held_ids), 2))
print("char-level held-out PPL per token:", round(sequence_ppl(bigram_probs(cc, 0.1), c_held), 2))

## Recap
- Perplexity $= e^{\text{mean NLL}}$ = geometric mean of $1/p$ = an effective number of equally likely choices.
- Nats or bits give the same perplexity, as long as you exponentiate with the matching base.
- Every model fits its training text better than held-out text, which is why perplexity is measured on held-out data.
- Perplexity is per token, so different tokenizers aren't comparable. Use bits per byte instead.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def perplexity(probs):
    probs = np.asarray(probs, dtype=float)
    return float(np.exp(-np.mean(np.log(probs))))

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def loss_to_ppl(loss_nats):
    return float(np.exp(loss_nats))

def bits_per_token(probs):
    return float(-np.mean(np.log2(np.asarray(probs, dtype=float))))

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def bigram_probs(counts, alpha):
    return (counts + alpha) / (counts.sum(axis=1, keepdims=True) + alpha * counts.shape[1])

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def sequence_ppl(P, ids):
    return perplexity(P[ids[:-1], ids[1:]])

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def per_token_ppl(total_nll_nats, n_tokens):
    return float(np.exp(total_nll_nats / n_tokens))

def bits_per_byte(total_nll_nats, n_bytes):
    return total_nll_nats / math.log(2) / n_bytes